# 01 — Data collection, cleaning and EDA

**Data:** Indian G-Sec yields at 7 tenors (3M, 1Y, 2Y, 5Y, 10Y, 15Y, 30Y), daily, 2012 → Nov 2026.
Source: investing.com historical data (manual CSV download, saved in `data/raw/IN_<tenor>.csv`).

**Features:** engineered from the yield curve itself (section 12 here, built in `02_models.ipynb`).

Output: `data/processed/dataset.csv`, used by `02_models.ipynb`. Figures are saved to `figures/`.

In [ ]:
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings("ignore")

RAW = Path("data/raw")
PROC = Path("data/processed")
FIG = Path("figures")
PROC.mkdir(parents=True, exist_ok=True)
FIG.mkdir(parents=True, exist_ok=True)

TENORS = ["3M", "1Y", "2Y", "5Y", "10Y", "15Y", "30Y"]
TENOR_YEARS = {"3M": 0.25, "1Y": 1, "2Y": 2, "5Y": 5, "10Y": 10, "15Y": 15, "30Y": 30}
START = "2012-01-01"

sns.set_theme(style="whitegrid")
plt.rcParams["figure.dpi"] = 110

def savefig(name):
    plt.savefig(FIG / f"{name}.png", bbox_inches="tight")

## 1. Load the yield CSVs

In [ ]:
def load_investing_csv(path):
    """investing.com export -> Series of daily closing yields (%), indexed by date."""
    df = pd.read_csv(path, thousands=",")
    df["Date"] = pd.to_datetime(df["Date"], format="%m/%d/%Y")
    return df.set_index("Date").sort_index()["Price"].astype(float)

raw = pd.DataFrame({t: load_investing_csv(RAW / f"IN_{t}.csv") for t in TENORS})
print(raw.shape)
print("Missing values per tenor:")
print(raw.isna().sum())
raw.describe().T

## 2. Cleaning

**2a. 3M price-quoted rows.** For 6–16 May 2025 the 3M file holds the T-bill *price* (≈98.6) instead of its yield.
A 91-day T-bill priced at P has yield ≈ `(100/P − 1) × 365/91 × 100`, but converting gives values that jump by
±30 bps from day to day (the exact days-to-maturity of the quoted bill is unknown), so we **drop** these 8 days instead.

In [ ]:
bad = raw["3M"] > 20
print(raw.loc[bad, "3M"])
print("Converted (for reference only):",
      ((100 / raw.loc[bad, "3M"] - 1) * 365 / 91 * 100).round(2).tolist())
bad_dates = raw.index[bad]
raw = raw.drop(bad_dates)

**2b. Calendar.** Until 2013 the G-Sec market also held short Saturday sessions. To get one consistent daily calendar
we keep Monday–Friday rows only, and only days on which the 10Y benchmark traded.
A missing tenor on such a day is forward-filled (at most 3 days), i.e. we use its last observed yield.

In [ ]:
print("Rows by weekday (0=Mon):", raw.index.dayofweek.value_counts().sort_index().to_dict())

yields = raw[(raw.index.dayofweek < 5) & raw["10Y"].notna()]
yields = yields.ffill(limit=3).dropna()
yields = yields[yields.index >= START]
print(yields.shape, yields.index.min().date(), "->", yields.index.max().date())

**2c. Large one-day moves.** A rolling-median (Hampel) check flags the points below.
Most of them are real market events (16 Jul 2013 RBI liquidity tightening during the taper tantrum,
22 May 2020 surprise repo cut, May 2022 surprise hike), so we **keep** them — removing them would remove exactly the moves
a model should learn about.

In [ ]:
def hampel_flags(s, window=11, min_move=0.40, n_mad=5):
    med = s.rolling(window, center=True, min_periods=3).median()
    dev = (s - med).abs()
    mad = dev.rolling(window, center=True, min_periods=3).median()
    return (dev > min_move) & (dev > n_mad * 1.4826 * mad)

flags = pd.DataFrame({t: hampel_flags(yields[t]) for t in TENORS})
for t in TENORS:
    for d in yields.index[flags[t]]:
        print(f"{t:>4}  {d.date()}  {yields.at[d, t]:.3f}")

## 3. Save the clean dataset

In [ ]:
data = yields.copy()
assert len(data) > 1000, f"Only {len(data)} rows - check the CSVs in data/raw"
data.to_csv(PROC / "dataset.csv")
print(data.shape, data.index.min().date(), "->", data.index.max().date())
data.tail()

# EDA

## 4. Summary statistics

In [ ]:
summary = data[TENORS].describe().T
summary["skew"] = data[TENORS].skew()
summary.round(3)

## 5. Yields over time

In [ ]:
events = {"2013-07-16": "Taper tantrum /\nRBI tightening",
          "2020-03-27": "COVID\nrate cut",
          "2022-05-04": "Hiking cycle\nbegins",
          "2025-02-07": "Easing cycle\nbegins"}

fig, ax = plt.subplots(figsize=(12, 5))
for t in TENORS:
    ax.plot(data.index, data[t], lw=1, label=t)
for d, label in events.items():
    ax.axvline(pd.Timestamp(d), color="grey", ls="--", lw=0.8)
    ax.text(pd.Timestamp(d), ax.get_ylim()[1] * 0.98, label, fontsize=8, va="top")
ax.set_ylabel("Yield (%)")
ax.set_title("Indian G-Sec yields by tenor")
ax.legend(ncol=7, loc="lower left", fontsize=8)
savefig("01_yields_over_time")
plt.show()

## 6. The yield curve on selected dates

In [ ]:
snap_dates = ["2013-08-29", "2016-01-04", "2020-06-01", "2023-01-02", str(data.index.max().date())]
x = [TENOR_YEARS[t] for t in TENORS]

fig, ax = plt.subplots(figsize=(9, 5))
for d in snap_dates:
    row = data.loc[data.index.asof(pd.Timestamp(d)), TENORS]
    ax.plot(x, row.values, marker="o", label=str(row.name.date()))
ax.set_xscale("log")
ax.set_xticks(x)
ax.set_xticklabels(TENORS)
ax.set_xlabel("Maturity (log scale)")
ax.set_ylabel("Yield (%)")
ax.set_title("Shape of the yield curve")
ax.legend()
savefig("02_curve_snapshots")
plt.show()

## 7. Slope of the curve (term spreads)

In [ ]:
spreads = pd.DataFrame({
    "10Y - 3M": data["10Y"] - data["3M"],
    "30Y - 10Y": data["30Y"] - data["10Y"],
})
ax = spreads.plot(figsize=(12, 4), lw=1)
ax.axhline(0, color="black", lw=0.8)
ax.set_ylabel("Spread (percentage points)")
ax.set_title("Term spreads (negative = inverted)")
savefig("03_term_spreads")
plt.show()
print("Days with inverted 10Y-3M:", int((spreads["10Y - 3M"] < 0).sum()))

## 8. Correlations: levels vs daily changes

Yield **levels** across tenors are almost perfectly correlated because they all trend together.
Daily **changes** are much less correlated, especially between the short end and the long end.

In [ ]:
levels = data[TENORS]
changes = data[TENORS].diff().dropna() * 100      # basis points

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
sns.heatmap(levels.corr(), annot=True, fmt=".2f", cmap="coolwarm", vmin=-1, vmax=1, ax=axes[0])
axes[0].set_title("Correlation of yield levels")
sns.heatmap(changes.corr(), annot=True, fmt=".2f", cmap="coolwarm", vmin=-1, vmax=1, ax=axes[1])
axes[1].set_title("Correlation of daily changes")
savefig("04_correlations")
plt.show()

## 9. Distribution of daily changes

In [ ]:
fig, ax = plt.subplots(figsize=(11, 4))
sns.boxplot(data=changes.clip(-50, 50), ax=ax, fliersize=1)
ax.set_ylabel("Daily change (bps), clipped to ±50")
ax.set_title("Daily yield changes by tenor")
savefig("05_daily_change_boxplot")
plt.show()

pd.DataFrame({"std_bps": changes.std(), "kurtosis": changes.kurt(),
              "max_abs_bps": changes.abs().max()}).round(2)

Excess kurtosis far above 0 means **fat tails**: most days barely move, but a few days move a lot.

## 10. Persistence: are yields predictable from yesterday?

ADF test: null hypothesis = the series has a unit root (behaves like a random walk).
- Levels: high p-value → cannot reject → today's yield ≈ yesterday's yield + noise.
- Changes: p ≈ 0 → stationary.

This is why, in `02_models`, we predict the **next-day change** and compare every model with the naive forecast
"tomorrow = today". A model predicting levels gets R² ≈ 0.99 simply by copying today's value.

In [ ]:
from statsmodels.tsa.stattools import adfuller
from statsmodels.graphics.tsaplots import plot_acf

adf = pd.DataFrame({
    "ADF p-value (level)": [adfuller(levels[t])[1] for t in TENORS],
    "ADF p-value (daily change)": [adfuller(changes[t])[1] for t in TENORS],
}, index=TENORS)
display(adf.round(4))

fig, axes = plt.subplots(1, 2, figsize=(12, 3.5))
plot_acf(levels["10Y"], lags=40, ax=axes[0], title="ACF — 10Y level")
plot_acf(changes["10Y"], lags=40, ax=axes[1], title="ACF — 10Y daily change")
savefig("06_acf_10y")
plt.show()

## 11. PCA: level, slope and curvature

PCA on standardised daily changes of all 7 tenors. In most bond markets the first three components are
**level** (all tenors move together), **slope** (short end vs long end) and **curvature** (middle vs both ends).

In [ ]:
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

Z = StandardScaler().fit_transform(changes)
pca = PCA().fit(Z)
explained = pd.Series(pca.explained_variance_ratio_, index=[f"PC{i+1}" for i in range(len(TENORS))])
print((explained.cumsum() * 100).round(1))

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
explained.plot.bar(ax=axes[0], color="steelblue")
axes[0].set_title("Explained variance ratio")
for i, name in enumerate(["PC1", "PC2", "PC3"]):
    axes[1].plot(TENORS, pca.components_[i], marker="o", label=name)
axes[1].axhline(0, color="black", lw=0.8)
axes[1].set_title("Loadings of first 3 components")
axes[1].legend()
savefig("07_pca")
plt.show()

## 12. Candidate features: do they say anything about tomorrow?

All model features are computed from the yield curve itself, using only data up to day *t*:

| Signal | Definition | Idea |
|---|---|---|
| `d1`, `d5`, `d20` | change over the last 1 / 5 / 20 days (bps) | momentum or reversal |
| `dev20` | yield minus its 20-day average (bps) | mean reversion |
| `vol20` | std. dev. of daily changes over 20 days (bps) | how turbulent the market is |
| `10Y−3M`, `30Y−10Y` | term spreads (pp) | slope of the curve |
| `curv` | 2×5Y − 2Y − 10Y (pp) | curvature |

The heatmap shows each signal's correlation with the **next-day change** of the same tenor.
Values close to zero show how hard next-day prediction is; this sets realistic expectations for `02_models`.

In [ ]:
TARGETS = ["3M", "5Y", "30Y"]
lead = {}
for t in TARGETS:
    y = data[t]
    d1 = y.diff() * 100
    signals = pd.DataFrame({
        "d1": d1,
        "d5": y.diff(5) * 100,
        "d20": y.diff(20) * 100,
        "dev20": (y - y.rolling(20).mean()) * 100,
        "vol20": d1.rolling(20).std(),
        "10Y-3M": data["10Y"] - data["3M"],
        "30Y-10Y": data["30Y"] - data["10Y"],
        "curv": 2 * data["5Y"] - data["2Y"] - data["10Y"],
    })
    next_change = y.diff().shift(-1) * 100
    lead[t] = signals.corrwith(next_change)

lead = pd.DataFrame(lead)
plt.figure(figsize=(6, 4.5))
sns.heatmap(lead, annot=True, fmt=".3f", cmap="coolwarm", center=0)
plt.title("Signal today vs yield change tomorrow (correlation)")
savefig("08_lead_correlations")
plt.show()